# Criteo Uplift Modeling: Individual Treatment Effect Estimation

**Single entry point.** Attach a Kaggle dataset containing `criteo-uplift-v2.1.csv`,
set `RUN_STAGE` below, then Run All. Every model, transform, and metric is
imported from `src/`; nothing is reimplemented in this notebook.

## Section 0 -- Introduction: problem and objective

### Problem statement

A standard predictive model answers:

> P(conversion | X) -- **"who is likely to convert?"**

That is not the question an advertiser actually needs answered. Some users
convert whether or not they see the ad; showing them the ad wastes budget.
Some users never convert no matter what. The only users worth targeting are
the ones whose behavior *changes because of* the treatment. That question is:

> tau(X) = P(Y=1 | T=1, X) - P(Y=1 | T=0, X) -- **"who changes behavior
> because of the treatment?"**

This is the **conditional average treatment effect (CATE)**, and estimating
it -- rather than a plain conversion probability -- is what **uplift
modeling** does. The distinction matters because a model that ranks users
well by P(convert | X) can rank them arbitrarily badly by tau(X); the two
quantities answer different questions and are not interchangeable.

### A simple example

| User | Converts without treatment | Converts with treatment | Interpretation |
|---|---|---|---|
| A | No | Yes | **Persuadable** -- the treatment caused the conversion. This is who targeting should reach. |
| B | Yes | Yes | Would have converted anyway -- treating them spends budget for no incremental gain. |
| C | No | No | Treatment had no effect -- not worth targeting. |

A response model (P(convert \| X)) cannot tell A and B apart -- both look
like likely converters. Only comparing outcomes *across* the treatment and
control arms can.

### Objective

Compare four estimators of increasing sophistication on **CRITEO-UPLIFTv2.1**
(~13.9M rows, a real advertising A/B test) and report which ranks users by
incremental conversion most effectively, under an evaluation protocol that
never lets a model touch the data it is ultimately scored on:

| Model | Idea |
|---|---|
| **Response model** (LightGBM) | Non-causal comparator: predicts `P(convert \| X)`, ignores treatment entirely |
| **T-Learner** | Two independent outcome models, one per arm; `tau_hat = mu1_hat - mu0_hat` |
| **X-Learner** | Cross-fitted pseudo-outcome regression; corrects the T-Learner's imbalanced-arm bias |
| **Causal Forest** (`econml.grf.CausalForest`) | Honest random forest that splits directly on treatment-effect heterogeneity |

### Evaluation

- **Qini curve / Qini above random** -- primary ranking statistic
- **AUUC** (area under the uplift curve) -- a differently-weighted second view
- **Uplift@K** and a **decile breakdown** -- coarser, more interpretable slices
- **CATE distribution** -- is the model finding real heterogeneity, or a near-constant effect?

See `README.md`'s "Methodology notes" for the modeling decisions that matter
(D32 categorical handling, X-Learner fold-local preprocessing, Causal Forest
categorical encoding) and the four development notebooks for the full
derivation of each estimator:

| Notebook | Covers |
|---|---|
| `01_data_processing` | data contract, feature semantics, the split |
| `02_baseline_models` | response model, and why it is *not* a causal estimator |
| `03_uplift_models` | T-Learner, X-Learner, the fold-local leakage fix |
| `04_causal_forest` | Causal Forest, categorical encoding, final comparison |

## Run parameters

- **`RUN_STAGE`** selects what to (re)compute this run. Every stage other than
  `"data"` first calls `ensure_data_artifacts()`, which loads existing data
  artifacts if they match the current `SAMPLE_ROWS`/`SEED`, or (re)computes
  them if missing or stale -- so `RUN_STAGE = "uplift"` alone works even on a
  fresh kernel, it just costs one data pass.

  | Value | Runs |
  |---|---|
  | `"data"` | data load, split, dataset/feature EDA |
  | `"baseline"` | response model |
  | `"uplift"` | T-Learner, X-Learner |
  | `"causal_forest"` | Causal Forest (the slow stage) |
  | `"report"` | final comparison + every visualization, **from artifacts only** -- fits nothing, so it is the stage to run after a kernel restart to see results without retraining |
  | `"all"` | every stage above, in order, in one session |
- **`SAMPLE_ROWS = None`** runs the complete ~13.9M-row experiment. Set e.g.
  `SAMPLE_ROWS = 1_000_000` for a fast end-to-end check first -- the sample is
  stratified on `(treatment, conversion)`, and every model sees the same rows.
- **`RUN_CAUSAL_FOREST = True`**: the Causal Forest is the bottleneck
  (`econml`'s `CausalForest` runs with `n_jobs=1`, required for reproducible
  predictions, so a full-data fit can take hours). Set `False` to skip it --
  the report stage still shows every other model and marks Causal Forest as
  pending rather than fabricate a result.

### Why this notebook runs as stages, not one long script

`econml`'s `CausalForest` one-hot-encodes categorical features into a dense
matrix (~76 columns at the shipped `K=8`); at full CRITEO scale that matrix
alone is several GB, on top of the LightGBM-transformed matrices the other
three models use. Holding all of it in one kernel at once is a real OOM risk
on a standard Kaggle kernel. This notebook instead runs as **independent,
restart-safe stages** -- each stage loads only what it needs, saves its
expensive output as an artifact under `outputs/kaggle_execution/`, and
releases memory (`del` + `gc.collect()`) before the next stage. Run All in
one session, or restart the kernel between stages (Data -> Baseline -> Uplift
-> Causal Forest -> Report) -- each stage picks up from the artifacts the
previous one saved, so nothing is retrained unnecessarily.

In [ ]:
RUN_STAGE = "all"          # "data" | "baseline" | "uplift" | "causal_forest" | "report" | "all"
SAMPLE_ROWS = None          # None = full dataset; e.g. 1_000_000 for a fast run
SEED = 42
RUN_CAUSAL_FOREST = True    # set False to skip the slowest stage

_VALID_STAGES = ("data", "baseline", "uplift", "causal_forest", "report", "all")
assert RUN_STAGE in _VALID_STAGES, f"RUN_STAGE={RUN_STAGE!r} must be one of {_VALID_STAGES}"

## Stage 0 -- Environment validation

Always runs, regardless of `RUN_STAGE`: locate the repository (this notebook
does not assume the kernel's working directory), make `src/` importable, and
verify every dependency actually imports before any expensive work starts.

In [ ]:
import sys, platform
from pathlib import Path


def find_repo_root() -> Path:
    bases = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]
    for base in bases:
        if not base.is_dir():
            continue
        if (base / "src" / "data.py").is_file():
            return base
        for child in sorted(p for p in base.iterdir() if p.is_dir()):
            if (child / "src" / "data.py").is_file():
                return child
    return None


REPO_ROOT = find_repo_root()

if REPO_ROOT is None and Path("/kaggle/working").is_dir():
    # Repo not attached -- try cloning it (requires Internet enabled in kernel settings).
    import subprocess
    target = Path("/kaggle/working/causal-uplift-modeling")
    print("Repository not found; attempting clone into", target)
    result = subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/arthur105204/causal-uplift-modeling.git", str(target)],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
    REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate the repository. Either attach it as a Kaggle dataset, "
        "clone it into /kaggle/working, or enable Internet in the kernel settings "
        "so this cell can clone it automatically."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("python     :", platform.python_version())
print("repo root  :", REPO_ROOT)

In [ ]:
import importlib

REQUIRED = ["numpy", "pandas", "sklearn", "lightgbm", "econml", "pyarrow", "matplotlib", "yaml", "joblib", "psutil"]
missing = []
for name in REQUIRED:
    try:
        module = importlib.import_module(name)
        print(f"{name:12s} {getattr(module, '__version__', 'n/a')}")
    except ImportError as exc:
        missing.append(name)
        print(f"{name:12s} MISSING ({exc})")

VERSION_PINS = {"lightgbm": "4.7.0", "econml": "0.17.0"}  # must match requirements.txt

if missing:
    print("\nInstalling missing packages...")
    import subprocess
    to_install = [f"{name}=={VERSION_PINS[name]}" if name in VERSION_PINS else name for name in missing]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=False)
    print("Re-run this cell to confirm.")
else:
    print("\nAll dependencies present.")

In [ ]:
import gc
import time

# Everything below comes from src/ -- no model code is defined in this notebook.
from src.data import (
    CATEGORICAL_FEATURES, CONTINUOUS_FEATURES, FEATURE_COLUMNS,
    PRIMARY_OUTCOME, SECONDARY_OUTCOME, TREATMENT_COLUMN,
    basic_summary, load_config, load_csv, load_parquet, on_kaggle,
    resolve_csv_path, save_parquet,
)
from src.preprocessing import (
    CausalForestCategoricalEncoder, LightGBMFeatureTransform, train_validation_test_split,
)
from src.models import (
    fit_causal_forest, fit_response_model, fit_t_learner, fit_x_learner,
    predict, predict_causal_forest_tau,
)
from src.evaluation import compute_ate, evaluate_ranking, response_diagnostics
from src.artifacts import (
    artifact_root, config_fingerprint, load_csv_artifact, load_json, load_pickle,
    save_csv, save_json, save_pickle, stage_dir,
)

CONFIG = load_config()
ARTIFACT_ROOT = artifact_root()
print("on kaggle     :", on_kaggle())
print("artifact root :", ARTIFACT_ROOT)
print("imports OK")

In [ ]:
# Reproducibility fingerprint -- what produced this run, in case a reviewer
# needs to match a result back to an exact config/commit.
import platform as _platform
import subprocess as _subprocess


def _git_commit() -> str:
    try:
        result = _subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT,
            capture_output=True, text=True, timeout=5,
        )
        return result.stdout.strip() or "unknown"
    except Exception:
        return "unknown (not a git checkout, or git unavailable)"


ENV_INFO = {
    "python_version": _platform.python_version(),
    "platform": _platform.platform(),
    "git_commit": _git_commit(),
    "seed": SEED,
    "sample_rows": SAMPLE_ROWS,
    "run_stage": RUN_STAGE,
}
for key, value in ENV_INFO.items():
    print(f"{key:15s}: {value}")

In [ ]:
# Shared helper: every model-producing stage (baseline / uplift / causal
# forest) packages its test-set evaluation identically -- one prediction
# artifact, one metrics artifact, one set of curve artifacts -- so the report
# stage can reconstruct the full comparison from disk without importing any
# stage-specific code. This is notebook I/O plumbing around
# src.evaluation.evaluate_ranking, not a reimplementation of it.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def package_ranking_artifacts(
    model_dir, label, val_metrics, test_metrics,
    val_scores, test_scores, T_val, Y_val, T_test, Y_test,
    row_id_val, row_id_test, runtime_seconds, extra_metrics=None,
):
    predictions = pd.concat([
        pd.DataFrame({
            "row_id": np.asarray(row_id_val), "partition": "validation",
            "score": np.asarray(val_scores, dtype=np.float64),
            "treatment": np.asarray(T_val, dtype=np.float64), "outcome": np.asarray(Y_val, dtype=np.float64),
        }),
        pd.DataFrame({
            "row_id": np.asarray(row_id_test), "partition": "test",
            "score": np.asarray(test_scores, dtype=np.float64),
            "treatment": np.asarray(T_test, dtype=np.float64), "outcome": np.asarray(Y_test, dtype=np.float64),
        }),
    ], ignore_index=True)
    save_parquet(predictions, model_dir / "predictions.parquet")
    save_csv(test_metrics.qini_curve, model_dir / "qini_curve.csv")
    save_csv(test_metrics.uplift_curve, model_dir / "uplift_curve.csv")
    save_csv(test_metrics.decile_table, model_dir / "decile_table.csv")
    metrics = {
        "label": label,
        "runtime_seconds": runtime_seconds,
        "n_test": test_metrics.n,
        "val_qini_above_random": val_metrics.qini_above_random,
        "val_auuc_above_random": val_metrics.auuc_above_random,
        "test_qini_above_random": test_metrics.qini_above_random,
        "test_auuc_above_random": test_metrics.auuc_above_random,
        "test_qini_area": test_metrics.qini_area,
        "test_auuc_area": test_metrics.auuc_area,
        "test_uplift_at_k": test_metrics.uplift_at_k,
    }
    if extra_metrics:
        metrics.update(extra_metrics)
    save_json(metrics, model_dir / "metrics.json")
    return metrics


def load_ranking_artifact(model_dir):
    return {
        "metrics": load_json(model_dir / "metrics.json"),
        "qini_curve": load_csv_artifact(model_dir / "qini_curve.csv"),
        "uplift_curve": load_csv_artifact(model_dir / "uplift_curve.csv"),
        "decile_table": load_csv_artifact(model_dir / "decile_table.csv"),
        "predictions": load_parquet(model_dir / "predictions.parquet"),
    }


def artifact_is_fresh(meta_path, expected_signature):
    if not meta_path.is_file():
        return False
    try:
        return load_json(meta_path).get("data_signature") == expected_signature
    except Exception:
        return False

## Section 1 -- Dataset Understanding (Stage 1 -- Data processing)

Loads the raw CSV (auto-discovering the attached Kaggle dataset slug),
verifies the causal contract (`X = f0..f11`, `T = treatment`, `Y = conversion`),
computes the population ATE, optionally subsamples, and produces the seeded
70/15/15 train/validation/test split every later stage reads. It also
computes the data-quality and randomization diagnostics used in Sections 2
and 3 below, in the same single pass over the data.

**The dataset is anonymized.** `f0`-`f11` carry no documented business
meaning -- there is no public mapping from e.g. `f3` to "age" or "customer
segment". Everything below characterizes their **statistical behavior, data
quality, and predictive usefulness**; it does not, and cannot, assign them a
real-world interpretation.

**Feature semantics (D32).** All twelve `f0`-`f11` columns are stored as
`float64`, but physical storage does not imply semantic type: `f0`, `f2`,
`f7`, `f10` are genuinely **continuous**; `f1`, `f3`, `f4`, `f5`, `f6`, `f8`,
`f9`, `f11` are **categorical numeric tokens** with no ordinal meaning --
treating them as ordered numbers (for modeling *or* for a naive numeric
"average" in a diagnostic table) would fabricate an ordering that doesn't
exist. `exposure` is post-assignment and is loaded only for the contract
check -- never as a feature.

**Artifacts written** under `outputs/data/`: `dataset_summary.json`,
`feature_summary.csv`, `categorical_top_categories.csv`,
`covariate_balance.csv`, `split_summary.json`, `run_config.json`, and
`train.parquet` / `validation.parquet` / `test.parquet` (the raw, unencoded
splits every other stage reads instead of re-parsing the CSV). Every other
stage calls `ensure_data_artifacts()` first and reuses these unchanged if
they match the current `SAMPLE_ROWS`/`SEED`.

In [ ]:
def ensure_data_artifacts():
    # Load-or-compute the data-stage artifacts. Returns the data signature
    # every downstream stage's own artifacts are checked against.
    data_dir = stage_dir("data")
    split_cfg = CONFIG["split"]
    signature = config_fingerprint(SAMPLE_ROWS, SEED, split_cfg, CONFIG["data"])
    run_config_path = data_dir / "run_config.json"
    parquet_paths = [data_dir / f"{name}.parquet" for name in ("train", "validation", "test")]

    if artifact_is_fresh(run_config_path, signature) and all(p.is_file() for p in parquet_paths):
        print(f"Using cached data artifacts ({data_dir}), signature={signature}")
        return signature

    print("Computing data artifacts (missing or stale) ...")
    stage_start = time.perf_counter()

    csv_path = resolve_csv_path()
    frame = load_csv(csv_path)

    missing_features = [f for f in FEATURE_COLUMNS if f not in frame.columns]
    assert not missing_features, f"missing features: {missing_features}"
    assert TREATMENT_COLUMN in frame.columns, f"missing treatment column {TREATMENT_COLUMN!r}"
    assert PRIMARY_OUTCOME in frame.columns, f"missing outcome column {PRIMARY_OUTCOME!r}"
    assert set(frame[TREATMENT_COLUMN].unique()) <= {0, 1}, "treatment must be binary"
    assert set(frame[PRIMARY_OUTCOME].unique()) <= {0, 1}, "conversion must be binary"

    summary = basic_summary(frame)
    ate = compute_ate(frame[TREATMENT_COLUMN], frame[PRIMARY_OUTCOME])
    by_treatment = frame.groupby(TREATMENT_COLUMN)[[PRIMARY_OUTCOME, SECONDARY_OUTCOME]].mean()

    if SAMPLE_ROWS is not None and SAMPLE_ROWS < len(frame):
        from sklearn.model_selection import train_test_split as _tts
        strata = frame[TREATMENT_COLUMN].astype(str) + "_" + frame[PRIMARY_OUTCOME].astype(str)
        frame, _ = _tts(frame, train_size=SAMPLE_ROWS, random_state=SEED, stratify=strata)
        frame = frame.reset_index(drop=True)

    # --- Randomized-experiment validation: is treatment independent of X? ---
    # Continuous features -> standardized mean difference (SMD); categorical
    # tokens -> total variation distance between the two arms' empirical
    # category-frequency distributions. TVD (not a numeric mean-difference)
    # is used for categoricals deliberately -- per D32, f1/f3/f4/f5/f6/f8/f9/f11
    # have no ordinal meaning, so averaging their raw codes would fabricate
    # the same kind of ordering bug D32 exists to prevent, even here in a
    # diagnostic table.
    treated_mask, control_mask = frame[TREATMENT_COLUMN] == 1, frame[TREATMENT_COLUMN] == 0
    balance_rows = []
    for f in CONTINUOUS_FEATURES:
        s = frame[f].astype("float64")
        m1, m0 = s[treated_mask].mean(), s[control_mask].mean()
        pooled_std = ((s[treated_mask].std() ** 2 + s[control_mask].std() ** 2) / 2) ** 0.5
        smd = float((m1 - m0) / pooled_std) if pooled_std > 0 else float("nan")
        balance_rows.append({"feature": f, "kind": "continuous", "metric": "standardized_mean_diff", "value": smd})
    for f in CATEGORICAL_FEATURES:
        s = frame[f].astype("float64")
        p1 = s[treated_mask].value_counts(normalize=True)
        p0 = s[control_mask].value_counts(normalize=True)
        idx = p1.index.union(p0.index)
        tvd = float(0.5 * (p1.reindex(idx, fill_value=0.0) - p0.reindex(idx, fill_value=0.0)).abs().sum())
        balance_rows.append({"feature": f, "kind": "categorical", "metric": "total_variation_distance", "value": tvd})
    save_csv(pd.DataFrame(balance_rows), data_dir / "covariate_balance.csv")

    train_frame, val_frame, test_frame = train_validation_test_split(
        frame,
        train_fraction=split_cfg["train_fraction"],
        validation_fraction=split_cfg["validation_fraction"],
        test_fraction=split_cfg["test_fraction"],
        seed=SEED,
    )
    keep_cols = list(FEATURE_COLUMNS) + [TREATMENT_COLUMN, PRIMARY_OUTCOME, SECONDARY_OUTCOME]
    partitions = {"train": train_frame, "validation": val_frame, "test": test_frame}
    split_summary = {}
    for name, part in partitions.items():
        out = part.loc[:, keep_cols].reset_index(drop=True)
        out.insert(0, "row_id", out.index.to_numpy())
        save_parquet(out, data_dir / f"{name}.parquet")
        split_summary[name] = {
            "n_rows": int(len(part)),
            "treatment_rate": float(part[TREATMENT_COLUMN].mean()),
            "conversion_rate": float(part[PRIMARY_OUTCOME].mean()),
            "visit_rate": float(part[SECONDARY_OUTCOME].mean()),
        }

    # --- Data quality + statistical profiling (train partition only) ---
    feature_rows = []
    for f in CONTINUOUS_FEATURES:
        s = train_frame[f].astype("float64")
        desc = s.describe(percentiles=[0.25, 0.5, 0.75])
        feature_rows.append({
            "feature": f, "kind": "continuous", "n_unique": int(s.nunique()),
            "mean": float(desc["mean"]), "std": float(desc["std"]), "min": float(desc["min"]),
            "p25": float(desc["25%"]), "p50": float(desc["50%"]), "p75": float(desc["75%"]), "max": float(desc["max"]),
            "skew": float(s.skew()), "top_category": None, "top_category_share": None,
        })
    top_category_rows = []
    for f in CATEGORICAL_FEATURES:
        vc = train_frame[f].astype("float64").value_counts()
        feature_rows.append({
            "feature": f, "kind": "categorical", "n_unique": int(vc.shape[0]),
            "mean": None, "std": None, "min": None, "p25": None, "p50": None, "p75": None, "max": None,
            "skew": None, "top_category": float(vc.index[0]), "top_category_share": float(vc.iloc[0] / len(train_frame)),
        })
        top10 = vc.head(10)
        for rank, (_, count) in enumerate(top10.items(), start=1):
            top_category_rows.append({
                "feature": f, "rank": rank, "count": int(count), "share": float(count / len(train_frame)),
            })
    save_csv(pd.DataFrame(feature_rows), data_dir / "feature_summary.csv")
    save_csv(pd.DataFrame(top_category_rows), data_dir / "categorical_top_categories.csv")

    dataset_summary = {
        "n_rows": summary["n_rows"], "n_cols": summary["n_cols"],
        "n_features": len(FEATURE_COLUMNS),
        "continuous_features": list(CONTINUOUS_FEATURES), "categorical_features": list(CATEGORICAL_FEATURES),
        "null_counts": {str(k): int(v) for k, v in summary["null_counts"].items()},
        "treatment_counts": {str(k): int(v) for k, v in summary["treatment_counts"].items()},
        "conversion_rate": summary["conversion_rate"],
        "visit_rate": float(frame[SECONDARY_OUTCOME].mean()),
        "conversion_rate_by_treatment": {str(k): float(v) for k, v in by_treatment[PRIMARY_OUTCOME].items()},
        "visit_rate_by_treatment": {str(k): float(v) for k, v in by_treatment[SECONDARY_OUTCOME].items()},
        "ate": {"ate": ate.ate, "se": ate.se, "ci_95_low": ate.ci_95_low, "ci_95_high": ate.ci_95_high,
                "relative_lift": ate.relative_lift},
        "csv_path": str(csv_path), "csv_size_mb": csv_path.stat().st_size / 1024**2,
        "sample_rows_used": None if SAMPLE_ROWS is None else int(len(frame)),
    }
    save_json(dataset_summary, data_dir / "dataset_summary.json")
    save_json(split_summary, data_dir / "split_summary.json")
    save_json({
        "data_signature": signature, "seed": SEED, "sample_rows": SAMPLE_ROWS,
        "split": split_cfg, "env": ENV_INFO,
        "runtime_seconds": time.perf_counter() - stage_start,
    }, run_config_path)

    del frame, train_frame, val_frame, test_frame, by_treatment
    gc.collect()
    print(f"Data artifacts written to {data_dir} in {time.perf_counter() - stage_start:.1f}s")
    return signature


DATA_SIGNATURE = ensure_data_artifacts()

In [ ]:
if RUN_STAGE in ("data", "all"):
    data_dir = stage_dir("data")
    dataset_summary = load_json(data_dir / "dataset_summary.json")
    split_summary = load_json(data_dir / "split_summary.json")

    dataset_summary_table = pd.DataFrame({
        "Item": ["Rows (full)", "Features (X)", "Treatment variable", "Outcome variable (primary)",
                 "Secondary outcome", "Conversion rate", "Treatment rate", "Population ATE"],
        "Value": [
            f"{dataset_summary['n_rows']:,}",
            f"{dataset_summary['n_features']} ({len(dataset_summary['continuous_features'])} continuous, "
            f"{len(dataset_summary['categorical_features'])} categorical)",
            TREATMENT_COLUMN, PRIMARY_OUTCOME, SECONDARY_OUTCOME,
            f"{dataset_summary['conversion_rate']:.5f}",
            f"{dataset_summary['treatment_counts'].get('1', 0) / dataset_summary['n_rows']:.3f}",
            f"{dataset_summary['ate']['ate']:.6f}  (95% CI {dataset_summary['ate']['ci_95_low']:.6f} .. "
            f"{dataset_summary['ate']['ci_95_high']:.6f})",
        ],
    }).set_index("Item")
    display(dataset_summary_table)

    split_table = pd.DataFrame({
        "rows": {k: v["n_rows"] for k, v in split_summary.items()},
        "treatment_rate": {k: v["treatment_rate"] for k, v in split_summary.items()},
        "conversion_rate": {k: v["conversion_rate"] for k, v in split_summary.items()},
        "visit_rate": {k: v["visit_rate"] for k, v in split_summary.items()},
    })
    print("\nPer-partition rates (train/validation/test):")
    display(split_table.round(5))
else:
    print(f"Data EDA skipped (RUN_STAGE={RUN_STAGE!r}); artifacts still ensured above.")

### Column roles

| Column | Type | Role |
|---|---|---|
| `f0`, `f2`, `f7`, `f10` | continuous, anonymized | predictor (X) |
| `f1`, `f3`, `f4`, `f5`, `f6`, `f8`, `f9`, `f11` | categorical token, anonymized | predictor (X) |
| `treatment` | binary | intervention (T) -- was the ad shown? |
| `conversion` | binary | **primary outcome (Y)** -- did the user convert? |
| `visit` | binary | secondary outcome -- did the user visit? (not modeled here) |
| `exposure` | binary, post-assignment | audit-only -- never enters X; would leak post-treatment information if used as a feature |

## Section 2 -- Data Quality and Statistical Exploration

Question this section answers: *is the data clean enough to model, and what
does each feature actually look like?*

In [ ]:
if RUN_STAGE in ("data", "all"):
    null_counts = pd.Series(dataset_summary["null_counts"], name="missing_count")
    total_missing = int(null_counts.sum())
    print(f"Missing values across all {dataset_summary['n_rows']:,} rows and {len(null_counts)} columns: {total_missing}")
    if total_missing == 0:
        print("No missing-value handling is required anywhere in the pipeline -- confirmed, not assumed.")
    else:
        display((null_counts[null_counts > 0] / dataset_summary["n_rows"]).rename("missing_share").to_frame())

### 2.1 Numerical feature profiling

The reader should be able to inspect feature ranges without reading code --
this table (computed on the full train partition, not a sample) is the
complete numerical summary.

In [ ]:
if RUN_STAGE in ("data", "all"):
    feature_summary = load_csv_artifact(data_dir / "feature_summary.csv")

    train_full = load_parquet(data_dir / "train.parquet")
    PLOT_SAMPLE_ROWS = min(len(train_full), 500_000)
    plot_sample = train_full.sample(n=PLOT_SAMPLE_ROWS, random_state=SEED) if PLOT_SAMPLE_ROWS < len(train_full) else train_full
    del train_full
    gc.collect()
    print(f"Plotting sample: {len(plot_sample):,} rows drawn from the train partition (tables below use the full train partition)")

    cont_summary = feature_summary[feature_summary["kind"] == "continuous"].set_index("feature")
    display(
        cont_summary[["min", "max", "mean", "std", "p50", "skew", "n_unique"]]
        .rename(columns={"p50": "median", "n_unique": "unique"})
        .round(4)
    )
    most_skewed = cont_summary["skew"].abs().idxmax()
    print(f"Interpretation: {most_skewed} has the largest |skew| ({cont_summary.loc[most_skewed, 'skew']:.2f}) "
          "among the continuous features -- a long tail in its distribution, not a business claim about what it measures.")

In [ ]:
if RUN_STAGE in ("data", "all"):
    fig, axes = plt.subplots(1, len(CONTINUOUS_FEATURES), figsize=(4 * len(CONTINUOUS_FEATURES), 3))
    for ax, f in zip(axes, CONTINUOUS_FEATURES):
        ax.hist(plot_sample[f].astype("float64"), bins=50, color="#2b6cb0")
        ax.set_title(f)
    fig.suptitle("Distribution of representative continuous features (500K-row train sample)")
    fig.tight_layout()
    plt.show()

### 2.2 Categorical feature profiling

Hundreds of categories per feature cannot be usefully plotted -- the exact
top-10 table (full train partition) carries the complete information; the
chart below is a readable illustration of one feature's shape, not a
replacement for the table.

In [ ]:
if RUN_STAGE in ("data", "all"):
    cat_summary = feature_summary[feature_summary["kind"] == "categorical"].set_index("feature")
    display(cat_summary[["n_unique", "top_category_share"]].round(4))

    top_categories = load_csv_artifact(data_dir / "categorical_top_categories.csv")
    most_concentrated = cat_summary["top_category_share"].idxmax()
    example_table = top_categories[top_categories["feature"] == most_concentrated].set_index("rank")
    print(f"Example -- top 10 categories for {most_concentrated} "
          f"(n_unique={int(cat_summary.loc[most_concentrated, 'n_unique'])}), full train partition:")
    display(example_table[["count", "share"]].round(6))
    print(f"Interpretation: the single most frequent category alone covers "
          f"{cat_summary.loc[most_concentrated, 'top_category_share']:.1%} of {most_concentrated} -- "
          "this kind of concentration, seen across most categorical features here, is why the Causal "
          "Forest's top-K + OTHER encoding (Section 4.4) can use a small K without discarding much signal.")

In [ ]:
if RUN_STAGE in ("data", "all"):
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, f in zip(axes.ravel(), CATEGORICAL_FEATURES):
        top = plot_sample[f].astype("float64").value_counts().head(10)
        ax.bar(range(len(top)), top.to_numpy(), color="#2b6cb0")
        ax.set_xticks([])
        ax.set_title(f"{f} (top 10 of {int(cat_summary.loc[f, 'n_unique'])})", fontsize=9)
    fig.suptitle("Category frequency shape, representative sample (exact counts in the table above)")
    fig.tight_layout()
    plt.show()

    del plot_sample
    gc.collect()

## Section 3 -- Randomized Experiment Validation

Every causal claim in this notebook rests on one assumption: **treatment
assignment was effectively random with respect to X.** This section shows
the evidence for that assumption rather than asserting it.

In [ ]:
if RUN_STAGE in ("data", "all"):
    treated_n = dataset_summary["treatment_counts"].get("1", 0)
    control_n = dataset_summary["treatment_counts"].get("0", 0)
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    ax.bar(["Control", "Treatment"], [control_n, treated_n], color=["#888888", "#2b6cb0"])
    ax.set_ylabel("Rows")
    ax.set_title("Treatment vs. control")
    for i, v in enumerate([control_n, treated_n]):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom")
    fig.tight_layout()
    plt.show()
    print(f"Treatment rate: {treated_n / (treated_n + control_n):.1%} treated / "
          f"{control_n / (treated_n + control_n):.1%} control -- an intentionally imbalanced split "
          "(more treated than control), which is exactly the imbalance the X-Learner exists to correct for.")

In [ ]:
if RUN_STAGE in ("data", "all"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, key, title in [
        (ax1, "conversion_rate_by_treatment", "Conversion rate by arm"),
        (ax2, "visit_rate_by_treatment", "Visit rate by arm"),
    ]:
        rates = dataset_summary[key]
        ax.bar(["Control", "Treatment"], [rates.get("0", 0.0), rates.get("1", 0.0)], color=["#888888", "#2b6cb0"])
        ax.set_title(title)
        ax.set_ylabel("Rate")
    fig.tight_layout()
    plt.show()

    overall_rate = dataset_summary["conversion_rate"]
    print(f"Overall conversion rate: {overall_rate:.3%}. This outcome is rare -- fewer than "
          f"{overall_rate:.1%} of users convert at all -- which is what makes uplift estimation hard: "
          "every model here is trying to detect a *difference between two already-small rates*, on a "
          "signal that is smaller still. A model that looks weak on this task may simply be up against "
          "a genuinely low signal-to-noise ratio, not a implementation flaw (see Section 7).")

### Covariate balance

If assignment was truly random, treated and control users should look
statistically similar on every pre-treatment feature. This is checked
per-feature, using a metric appropriate to each feature's actual type (per
D32): a **standardized mean difference** for the four continuous features,
and a **total variation distance** between category-frequency distributions
for the eight categorical tokens -- never a numeric mean on a categorical
code, which would fabricate an ordering that isn't there. Values well under
~0.1 (a common SMD rule of thumb) support the randomization assumption;
large values would be a red flag for confounding.

In [ ]:
if RUN_STAGE in ("data", "all"):
    balance = load_csv_artifact(data_dir / "covariate_balance.csv")
    display(balance.round(4))

    worst = balance.loc[balance["value"].abs().idxmax()]
    all_small = (balance["value"].abs() < 0.1).all()
    verdict = "supports" if all_small else "does NOT clearly support"
    print(f"Largest imbalance: {worst['feature']} ({worst['metric']} = {worst['value']:.4f}). "
          f"Every feature is well under the 0.1 SMD/TVD guideline this run, which {verdict} the assumption "
          "that treatment assignment was independent of X -- the precondition every model below relies on.")

## Section 4 -- Modeling Strategy

Four estimators, in order of increasing causal sophistication. Each
subsection below states the model's **purpose**, what it actually computes,
and its **role** in the comparison -- read this before the results, not
after, since a result only means something in light of what the model was
built to do.

### 4.1 Response LightGBM (Stage 2)

**Purpose:** predict `P(conversion | X)`, ignoring treatment entirely.

**Role:** the **non-causal comparator**. It answers "who is likely to
convert?", not "who converts *because of* the treatment?" -- strong
predictive performance here says nothing about causal ranking quality. It
exists specifically so the causal estimators below have something honest to
be compared against; if a plain response model matches or beats them on the
uplift metrics, that is informative (see Section 7), not a failure to report.

**Artifacts written** under `outputs/baseline/`: `model.pkl`, `metrics.json`,
`predictions.parquet` (validation + test scores), `qini_curve.csv`,
`uplift_curve.csv`, `decile_table.csv`, and `roc_curve.csv` / `pr_curve.csv`
/ `calibration_curve.csv` (validation-set diagnostics -- meaningful here
because this model's output is a calibrated probability; the uplift models'
outputs are not).

In [ ]:
def ensure_lgbm_transform():
    # Load-or-fit the LightGBM categorical-feature transform. Used by the
    # response model and the T-Learner; the X-Learner fits its own fold-local
    # transforms internally and does not use this one (see Stage 3).
    prep_dir = stage_dir("preprocessing")
    meta_path = prep_dir / "lgbm_transform_metadata.json"
    transform_path = prep_dir / "lgbm_transform.joblib"
    if artifact_is_fresh(meta_path, DATA_SIGNATURE) and transform_path.is_file():
        return load_pickle(transform_path)

    train_frame = load_parquet(stage_dir("data") / "train.parquet")
    transform = LightGBMFeatureTransform().fit(train_frame)
    save_pickle(transform, transform_path)
    save_json({
        "data_signature": DATA_SIGNATURE,
        "continuous_features": list(CONTINUOUS_FEATURES),
        "categorical_features": list(CATEGORICAL_FEATURES),
    }, meta_path)
    del train_frame
    gc.collect()
    return transform

In [ ]:
def ensure_baseline_artifacts():
    baseline_dir = stage_dir("baseline")
    meta_path = baseline_dir / "metrics.json"
    if artifact_is_fresh(meta_path, DATA_SIGNATURE) and (baseline_dir / "model.pkl").is_file():
        print(f"Using cached baseline artifacts ({baseline_dir})")
        return load_json(meta_path)

    print("Fitting response model (cache missing or stale) ...")
    data_dir = stage_dir("data")
    transform = ensure_lgbm_transform()
    train_frame = load_parquet(data_dir / "train.parquet")
    val_frame = load_parquet(data_dir / "validation.parquet")
    test_frame = load_parquet(data_dir / "test.parquet")

    X_train, X_val, X_test = (transform.transform(f) for f in (train_frame, val_frame, test_frame))
    Y_train, Y_val, Y_test = (f[PRIMARY_OUTCOME] for f in (train_frame, val_frame, test_frame))
    T_train, T_val, T_test = (f[TREATMENT_COLUMN] for f in (train_frame, val_frame, test_frame))
    row_id_val, row_id_test = val_frame["row_id"], test_frame["row_id"]
    del train_frame
    gc.collect()

    start = time.perf_counter()
    model = fit_response_model(X_train, Y_train, X_val, Y_val, seed=SEED)
    runtime = time.perf_counter() - start
    print(f"fitted in {runtime:.1f}s (best iteration {model.best_iteration})")

    val_scores = predict(model, X_val)
    test_scores = predict(model, X_test)
    diagnostics = response_diagnostics(val_scores, Y_val)
    val_metrics = evaluate_ranking(val_scores, T_val, Y_val)
    test_metrics = evaluate_ranking(test_scores, T_test, Y_test)

    from sklearn.calibration import calibration_curve
    from sklearn.metrics import precision_recall_curve, roc_curve
    fpr, tpr, _ = roc_curve(Y_val, val_scores)
    precision, recall, _ = precision_recall_curve(Y_val, val_scores)
    frac_pos, mean_pred = calibration_curve(Y_val, val_scores, n_bins=10, strategy="quantile")
    save_csv(pd.DataFrame({"fpr": fpr, "tpr": tpr}), baseline_dir / "roc_curve.csv")
    save_csv(pd.DataFrame({"precision": precision, "recall": recall}), baseline_dir / "pr_curve.csv")
    save_csv(pd.DataFrame({"mean_predicted": mean_pred, "fraction_positive": frac_pos}),
              baseline_dir / "calibration_curve.csv")

    save_pickle(model, baseline_dir / "model.pkl")
    metrics = package_ranking_artifacts(
        baseline_dir, "Response LightGBM", val_metrics, test_metrics,
        val_scores, test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, runtime,
        extra_metrics={
            "data_signature": DATA_SIGNATURE, "best_iteration": int(model.best_iteration),
            "val_roc_auc": diagnostics.roc_auc, "val_average_precision": diagnostics.average_precision,
            "val_log_loss": diagnostics.log_loss,
        },
    )
    print("validation qini_above_random:", round(val_metrics.qini_above_random, 4))

    del X_train, X_val, X_test, val_frame, test_frame
    gc.collect()
    return metrics


if RUN_STAGE in ("baseline", "all"):
    BASELINE_METRICS = ensure_baseline_artifacts()
else:
    print(f"Baseline stage skipped (RUN_STAGE={RUN_STAGE!r})")

In [ ]:
if RUN_STAGE in ("baseline", "all"):
    baseline_dir = stage_dir("baseline")
    roc = load_csv_artifact(baseline_dir / "roc_curve.csv")
    pr = load_csv_artifact(baseline_dir / "pr_curve.csv")
    cal = load_csv_artifact(baseline_dir / "calibration_curve.csv")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(roc["fpr"], roc["tpr"], color="#2b6cb0")
    axes[0].plot([0, 1], [0, 1], "--", color="#888888")
    axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
    axes[0].set_title(f"ROC (AUC={BASELINE_METRICS['val_roc_auc']:.4f})")

    axes[1].plot(pr["recall"], pr["precision"], color="#2b6cb0")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title(f"PR (AP={BASELINE_METRICS['val_average_precision']:.4f})")

    axes[2].plot(cal["mean_predicted"], cal["fraction_positive"], "o-", color="#2b6cb0")
    axes[2].plot([0, 1], [0, 1], "--", color="#888888")
    axes[2].set_xlabel("Mean predicted probability"); axes[2].set_ylabel("Observed conversion rate")
    axes[2].set_title("Calibration (validation, 10 quantile bins)")
    fig.suptitle("Response model diagnostics -- validation partition")
    fig.tight_layout()
    plt.show()
    print("These three diagnostics are specific to Response LightGBM: it outputs a calibrated "
          "probability, so ROC/PR/calibration are meaningful. The uplift models below output a "
          "*difference* of two predictions, which these particular diagnostics do not apply to -- "
          "they are evaluated with Qini/AUUC/Uplift@K instead (Section 6).")

### 4.2 T-Learner (Stage 3)

**Purpose:** estimate CATE by fitting two *independent* outcome models, one
per arm -- `mu1_hat(X)` on treated rows only, `mu0_hat(X)` on control rows
only -- then subtracting: `tau_hat(X) = mu1_hat(X) - mu0_hat(X)`.

**Role:** the simplest genuinely causal estimator here. Its weakness is
exactly the imbalance shown in Section 3: with a 85/15 treatment/control
split, `mu0_hat` is trained on far fewer rows than `mu1_hat`, so the two
models can have very different bias/variance characteristics -- their
difference inherits both.

### 4.3 X-Learner (Stage 3)

**Purpose:** correct the T-Learner's imbalanced-arm weakness. It cross-fits
the two arm models (fold-local, see `README.md`'s Methodology notes for the
leakage bug this fixed), then constructs **pseudo-outcomes** -- for a treated
row, `D1 = Y - mu0_hat(X)` (the effect implied by what *would* have happened
under control); for a control row, `D0 = mu1_hat(X) - Y`. Two further models
are fit to predict `D1` and `D0` from X, and their weighted combination
(weighted by the empirical treatment rate `g`) is the final `tau_hat(X)`.

**Role:** designed for exactly this dataset's shape -- imbalanced arms, one
of them (control) comparatively data-starved. It should out-perform the
T-Learner *if* the imbalance was the T-Learner's binding constraint; if not,
the extra complexity buys nothing.

**Artifacts written** under `outputs/uplift/tlearner/` and
`outputs/uplift/xlearner/`, each with the same `model.pkl` / `metrics.json` /
`predictions.parquet` / `qini_curve.csv` / `uplift_curve.csv` /
`decile_table.csv` shape as the baseline stage.

In [ ]:
def ensure_uplift_artifacts():
    uplift_dir = stage_dir("uplift")
    tlearner_dir, xlearner_dir = uplift_dir / "tlearner", uplift_dir / "xlearner"
    if (artifact_is_fresh(tlearner_dir / "metrics.json", DATA_SIGNATURE) and (tlearner_dir / "model.pkl").is_file()
            and artifact_is_fresh(xlearner_dir / "metrics.json", DATA_SIGNATURE) and (xlearner_dir / "model.pkl").is_file()):
        print(f"Using cached uplift artifacts ({uplift_dir})")
        return load_json(tlearner_dir / "metrics.json"), load_json(xlearner_dir / "metrics.json")

    print("Fitting T-Learner / X-Learner (cache missing or stale) ...")
    data_dir = stage_dir("data")
    transform = ensure_lgbm_transform()
    train_frame = load_parquet(data_dir / "train.parquet")
    val_frame = load_parquet(data_dir / "validation.parquet")
    test_frame = load_parquet(data_dir / "test.parquet")

    X_train, X_val, X_test = (transform.transform(f) for f in (train_frame, val_frame, test_frame))
    Y_train, Y_val, Y_test = (f[PRIMARY_OUTCOME] for f in (train_frame, val_frame, test_frame))
    T_train, T_val, T_test = (f[TREATMENT_COLUMN] for f in (train_frame, val_frame, test_frame))
    row_id_val, row_id_test = val_frame["row_id"], test_frame["row_id"]

    tlearner_dir.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    t_learner = fit_t_learner(X_train, T_train, Y_train, X_val, T_val, Y_val, seed=SEED)
    t_runtime = time.perf_counter() - start
    print(f"T-Learner fitted in {t_runtime:.1f}s")
    t_val_scores, t_test_scores = t_learner.predict_tau(X_val), t_learner.predict_tau(X_test)
    t_val_metrics = evaluate_ranking(t_val_scores, T_val, Y_val)
    t_test_metrics = evaluate_ranking(t_test_scores, T_test, Y_test)
    save_pickle(t_learner, tlearner_dir / "model.pkl")
    t_metrics = package_ranking_artifacts(
        tlearner_dir, "T-Learner", t_val_metrics, t_test_metrics,
        t_val_scores, t_test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, t_runtime, extra_metrics={"data_signature": DATA_SIGNATURE},
    )
    print("validation qini_above_random:", round(t_val_metrics.qini_above_random, 4))

    del X_train, X_val, X_test
    gc.collect()

    xlearner_dir.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    x_learner = fit_x_learner(train_frame, T_train, Y_train, seed=SEED)
    x_runtime = time.perf_counter() - start
    print(f"X-Learner fitted in {x_runtime:.1f}s")
    X_val_x, X_test_x = transform.transform(val_frame), transform.transform(test_frame)
    x_val_scores, x_test_scores = x_learner.predict_tau(X_val_x), x_learner.predict_tau(X_test_x)
    x_val_metrics = evaluate_ranking(x_val_scores, T_val, Y_val)
    x_test_metrics = evaluate_ranking(x_test_scores, T_test, Y_test)
    save_pickle(x_learner, xlearner_dir / "model.pkl")
    x_metrics = package_ranking_artifacts(
        xlearner_dir, "X-Learner", x_val_metrics, x_test_metrics,
        x_val_scores, x_test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, x_runtime, extra_metrics={"data_signature": DATA_SIGNATURE},
    )
    print("validation qini_above_random:", round(x_val_metrics.qini_above_random, 4))

    del train_frame, val_frame, test_frame, X_val_x, X_test_x
    gc.collect()
    return t_metrics, x_metrics


if RUN_STAGE in ("uplift", "all"):
    TLEARNER_METRICS, XLEARNER_METRICS = ensure_uplift_artifacts()
else:
    print(f"Uplift stage skipped (RUN_STAGE={RUN_STAGE!r})")

### 4.4 Causal Forest (Stage 4)

**Purpose:** an *honest* random forest (`econml.grf.CausalForest`) whose
splits are chosen to directly maximize treatment-effect **heterogeneity**
between the resulting child nodes, rather than outcome-prediction accuracy.
Concretely, each split searches for the partition of X that makes the
treated-vs-control outcome difference *most different* between the two
child nodes -- the forest is explicitly hunting for subgroups whose response
to treatment differs, not merely for subgroups with different average
outcomes.

**Role:** the most flexible estimator in the comparison -- it makes no
assumption about how tau(X) varies with X (unlike the two-model-difference
structure of the T/X-Learners). This flexibility is not automatically
"better": it comes with a coarser categorical representation (Section 5) and
higher estimator variance, and is exactly the property that makes it the
most memory- and compute-intensive model to fit at this row count. Whether
that flexibility pays off empirically is a Section 7 question, not assumed
here.

`econml.grf.CausalForest` has no native categorical support and consumes a
dense numeric matrix, so categorical features go through
`CausalForestCategoricalEncoder`: **frequency-capped top-K one-hot**, with an
`OTHER` bucket for the tail and for categories unseen at train time (Section
2.2 showed why a small K still keeps most of the signal). Raw integer tokens
would fabricate a false ordering the forest's splits would exploit --
`fit_causal_forest` rejects that outright.

**This is the slow, memory-heavy stage** (`n_jobs=1`, required for
reproducible predictions; `max_depth=20`, a memory safety cap -- see
`configs/config.yaml`). The encoded train matrix is built, converted to the
array representation the forest actually consumes, and freed before the
validation/test matrices are ever built -- at most one CF-encoded partition
exists in memory at a time. `K = configs/config.yaml:
causal_forest.categorical_top_k` (shipped default `8`) is a resource-ladder
choice, not a free parameter -- see README's Methodology notes for why `K=32`
risks ~21GB for the encoded train matrix alone at full CRITEO scale.

**Artifacts written** under `outputs/causal_forest/`: the usual
`model.pkl` / `metrics.json` / prediction and curve files, plus
`resource_evidence.json` recording the actual `K`, `max_depth`, encoded
column count, and an estimated encoded-matrix size -- the resource-gate
evidence for whichever config this run actually used.

In [ ]:
import threading

import psutil

try:
    import resource  # POSIX only -- Kaggle kernels are Linux; absent on Windows dev machines.
except ImportError:
    resource = None


def _peak_rss_mb():
    """OS-reported peak resident set size so far, in MB (Linux ru_maxrss is KB)."""
    if resource is None:
        return None
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


class _ResourceWatchdog:
    """Periodically persists peak RSS to disk during a long fit() call, so the
    last observed value survives a hard OOM kernel restart -- which kills the
    process before any in-memory evidence can be written."""

    def __init__(self, path, interval_seconds=30.0):
        self._path = path
        self._interval = interval_seconds
        self._stop_event = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _snapshot(self):
        peak = _peak_rss_mb()
        if peak is not None:
            save_json({"timestamp": time.time(), "peak_rss_mb": peak}, self._path)

    def _run(self):
        while not self._stop_event.wait(self._interval):
            self._snapshot()

    def start(self):
        if resource is not None:
            self._thread.start()
        return self

    def stop(self):
        self._stop_event.set()
        if self._thread.is_alive():
            self._thread.join(timeout=5)
        self._snapshot()


_MEMORY_PROCESS = psutil.Process()


def _log_memory(tag: str, log: list) -> float:
    """Point-in-time process RSS, in GB -- logged at named checkpoints along
    the Causal Forest path (encoder fit/transform -> fit_causal_forest ->
    predict) to identify which stage actually drives peak memory, instead of
    guessing from the encoded-matrix size alone."""
    gb = _MEMORY_PROCESS.memory_info().rss / 1024**3
    print(f"[MEMORY] {tag}: {gb:.3f} GB")
    log.append({"tag": tag, "rss_gb": gb, "timestamp": time.time()})
    return gb

In [ ]:
def ensure_cf_encoder():
    prep_dir = stage_dir("preprocessing")
    meta_path = prep_dir / "cf_encoder_metadata.json"
    encoder_path = prep_dir / "cf_encoder.joblib"
    k = CONFIG["causal_forest"]["categorical_top_k"]
    if artifact_is_fresh(meta_path, DATA_SIGNATURE) and encoder_path.is_file():
        if load_json(meta_path).get("k") == k:
            return load_pickle(encoder_path)

    train_frame = load_parquet(stage_dir("data") / "train.parquet")
    encoder = CausalForestCategoricalEncoder(k=k).fit(train_frame)
    save_pickle(encoder, encoder_path)
    save_json({"data_signature": DATA_SIGNATURE, "k": k}, meta_path)
    del train_frame
    gc.collect()
    return encoder

In [ ]:
def ensure_causal_forest_artifacts():
    cf_dir = stage_dir("causal_forest")
    meta_path = cf_dir / "metrics.json"
    if artifact_is_fresh(meta_path, DATA_SIGNATURE) and (cf_dir / "model.pkl").is_file():
        print(f"Using cached Causal Forest artifacts ({cf_dir})")
        return load_json(meta_path)

    print("Fitting Causal Forest (cache missing or stale; slow stage, n_jobs=1) ...")
    memory_log: list = []
    data_dir = stage_dir("data")
    cf_config = CONFIG["causal_forest"]
    encoder = ensure_cf_encoder()

    _log_memory("Before loading CF training data", memory_log)
    train_frame = load_parquet(data_dir / "train.parquet")
    _log_memory("After loading train parquet", memory_log)
    T_train, Y_train = train_frame[TREATMENT_COLUMN], train_frame[PRIMARY_OUTCOME]
    X_train_cf = encoder.transform(train_frame)
    encoded_feature_count = int(X_train_cf.shape[1])
    n_train_rows = int(len(X_train_cf))
    _log_memory("After categorical encoding", memory_log)
    del train_frame
    gc.collect()

    # Convert to the ndarray fit_causal_forest actually consumes and release
    # the DataFrame first -- otherwise both representations of the encoded
    # train matrix (DataFrame + ndarray) stay resident for the entire
    # multi-hour, n_jobs=1 fit.
    X_train_arr = X_train_cf.to_numpy()
    del X_train_cf
    gc.collect()

    # Evidence written BEFORE fit() -- resource_evidence.json below is only written
    # after fit() returns, so it is lost if the kernel OOM-restarts mid-fit. This
    # file and the watchdog snapshot survive on disk regardless.
    pre_fit_evidence = {
        "dtype": str(X_train_arr.dtype),
        "shape": list(X_train_arr.shape),
        "bytes": int(X_train_arr.nbytes),
        "estimated_gb": X_train_arr.nbytes / 1024**3,
        "timestamp": time.time(),
        "n_estimators": cf_config["n_estimators"],
        # .get(): max_depth/max_features were added to configs/config.yaml after
        # this notebook's other causal_forest fields shipped -- an older
        # config.yaml (e.g. a Kaggle checkout tracking a commit before either was
        # added) would otherwise KeyError here, before the fit even starts. Both
        # are resource-evidence *reporting* fields only -- the actual fit always
        # uses src.models.CAUSAL_FOREST_CONFIG, independent of config.yaml.
        "max_depth": cf_config.get("max_depth"),
        "max_features": cf_config.get("max_features"),
        "honest": cf_config["honest"],
        "inference": cf_config["inference"],
        "n_jobs": cf_config["n_jobs"],
    }
    save_json(pre_fit_evidence, cf_dir / "pre_fit_resource_evidence.json")
    print(f"pre-fit X: dtype={pre_fit_evidence['dtype']} shape={pre_fit_evidence['shape']} "
          f"~{pre_fit_evidence['estimated_gb']:.3f} GB")

    _log_memory("Before Causal Forest fitting", memory_log)
    watchdog = _ResourceWatchdog(cf_dir / "resource_watchdog.json").start()
    start = time.perf_counter()
    model = fit_causal_forest(X_train_arr, T_train, Y_train, seed=SEED)
    runtime = time.perf_counter() - start
    watchdog.stop()
    print(f"fitted in {runtime:.1f}s")
    del X_train_arr
    gc.collect()
    _log_memory("After Causal Forest fitting", memory_log)

    val_frame = load_parquet(data_dir / "validation.parquet")
    T_val, Y_val, row_id_val = val_frame[TREATMENT_COLUMN], val_frame[PRIMARY_OUTCOME], val_frame["row_id"]
    X_val_cf = encoder.transform(val_frame)
    _log_memory("Before validation prediction", memory_log)
    val_scores = predict_causal_forest_tau(model, X_val_cf)
    val_metrics = evaluate_ranking(val_scores, T_val, Y_val)
    del val_frame, X_val_cf
    gc.collect()

    test_frame = load_parquet(data_dir / "test.parquet")
    T_test, Y_test, row_id_test = test_frame[TREATMENT_COLUMN], test_frame[PRIMARY_OUTCOME], test_frame["row_id"]
    X_test_cf = encoder.transform(test_frame)
    _log_memory("Before test prediction", memory_log)
    test_scores = predict_causal_forest_tau(model, X_test_cf)
    test_metrics = evaluate_ranking(test_scores, T_test, Y_test)
    del test_frame, X_test_cf
    gc.collect()

    save_json({"checkpoints": memory_log}, cf_dir / "memory_checkpoints.json")

    save_pickle(model, cf_dir / "model.pkl")
    save_json({
        "categorical_top_k": cf_config["categorical_top_k"],
        "encoded_feature_count": encoded_feature_count,
        "n_train_rows": n_train_rows,
        "dtype": pre_fit_evidence["dtype"],
        "estimated_encoded_matrix_bytes": pre_fit_evidence["bytes"],
        "n_estimators": cf_config["n_estimators"],
        # .get(): same stale-config-yaml risk as pre_fit_evidence above.
        "max_depth": cf_config.get("max_depth"), "max_features": cf_config.get("max_features"),
        "honest": cf_config["honest"],
        "min_samples_leaf": cf_config["min_samples_leaf"], "max_samples": cf_config["max_samples"],
        "subforest_size": cf_config["subforest_size"], "n_jobs": cf_config["n_jobs"],
        "runtime_seconds": runtime,
    }, cf_dir / "resource_evidence.json")

    metrics = package_ranking_artifacts(
        cf_dir, "Causal Forest", val_metrics, test_metrics,
        val_scores, test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, runtime, extra_metrics={"data_signature": DATA_SIGNATURE},
    )
    print("validation qini_above_random:", round(val_metrics.qini_above_random, 4))
    return metrics


if RUN_STAGE in ("causal_forest", "all"):
    if RUN_CAUSAL_FOREST:
        CAUSAL_FOREST_METRICS = ensure_causal_forest_artifacts()
    else:
        CAUSAL_FOREST_METRICS = None
        print("Causal Forest skipped (RUN_CAUSAL_FOREST = False)")
else:
    print(f"Causal Forest stage skipped (RUN_STAGE={RUN_STAGE!r})")

In [ ]:
if RUN_STAGE in ("causal_forest", "all") and RUN_CAUSAL_FOREST:
    resource_evidence = load_json(stage_dir("causal_forest") / "resource_evidence.json")
    gb = resource_evidence["estimated_encoded_matrix_bytes"] / 1024**3
    print(f"K (categorical_top_k)   : {resource_evidence['categorical_top_k']}")
    print(f"Encoded feature count   : {resource_evidence['encoded_feature_count']}")
    print(f"Train rows encoded      : {resource_evidence['n_train_rows']:,}")
    dtype_display = resource_evidence.get("dtype", "float64")  # pre-float32-fix cached artifacts had no dtype field
    print(f"Estimated encoded matrix: {gb:.3f} GB ({dtype_display}, before CausalForest's own fit-time overhead)")
    max_depth_display = resource_evidence.get("max_depth") or "n/a (older run/config)"
    max_features_display = resource_evidence.get("max_features") or "n/a (older run/config)"
    print(f"n_estimators={resource_evidence['n_estimators']}  max_depth={max_depth_display}  "
          f"max_features={max_features_display}  honest={resource_evidence['honest']}  "
          f"n_jobs={resource_evidence['n_jobs']}  runtime={resource_evidence['runtime_seconds']:.1f}s")

## Section 5 -- Model Training Results (Stage 5 -- Final report)

**Fits nothing below.** Every cell in Sections 5-9 reads only the artifacts
each earlier stage saved under `outputs/` -- this is the stage to run
(`RUN_STAGE = "report"`) after a kernel restart to see the full comparison
without retraining anything. Whichever of the four models were actually run
appear below; a model whose stage was skipped or never run is shown as
unavailable, never fabricated.

The table below is a training summary, not a parameter dump -- full
configuration for technical readers is in the Appendix at the end of this
notebook.

In [ ]:
if RUN_STAGE in ("report", "all"):
    MODEL_STAGE_DIRS = {
        "Response LightGBM": stage_dir("baseline"),
        "T-Learner": stage_dir("uplift") / "tlearner",
        "X-Learner": stage_dir("uplift") / "xlearner",
        "Causal Forest": stage_dir("causal_forest"),
    }
    available = {label: d for label, d in MODEL_STAGE_DIRS.items() if (d / "metrics.json").is_file()}
    print("Available model artifacts:", list(available) or "NONE")
    if not available:
        raise RuntimeError(
            "No model artifacts found under outputs/. Run RUN_STAGE in "
            "{'baseline', 'uplift', 'causal_forest', 'all'} at least once first."
        )

    figures_dir = stage_dir("report") / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

    IMPLEMENTATION_NOTES = {
        "Response LightGBM": "single LightGBM classifier, early-stopped on validation",
        "T-Learner": "two independent LightGBM classifiers (one per arm)",
        "X-Learner": "2-fold cross-fitted pseudo-outcome regression, fold-local preprocessing",
        "Causal Forest": (f"econml CausalForest, n_estimators={CONFIG['causal_forest']['n_estimators']}, "
                           f"max_depth={CONFIG['causal_forest'].get('max_depth', 'n/a')}, honest=True, n_jobs=1"),
    }
    training_summary = pd.DataFrame([
        {
            "model": label,
            "runtime_seconds": round(load_json(d / "metrics.json")["runtime_seconds"], 1),
            "implementation": IMPLEMENTATION_NOTES[label],
        }
        for label, d in available.items()
    ]).set_index("model")
    display(training_summary)

## Section 6 -- Evaluation Metrics

Everything below is scored on the **test** partition -- never fit on,
early-stopped against, or used for selection. Before each metric: what
question does it actually answer?

**Qini above random** -- *does ranking users by this model's predicted
uplift, and targeting the top of that ranking, produce more incremental
conversions than targeting the same number of users at random?* This is the
primary statistic used to compare models below. A theoretical random-ranking
reference is included as the honest floor: an uplift model that cannot beat
it has not earned its complexity.

**AUUC above random** -- the same question, weighted differently (by
response-*rate* difference rather than Qini's count-ratio reweighting). It
is reported as a second view because Qini and AUUC can occasionally disagree
under strong arm imbalance, and a ranking that holds up under both is more
trustworthy than one that only looks good under one.

**Uplift@K** -- *among the top K% of users by predicted uplift, how much
incremental response was actually observed?* This is the coarsest, most
operationally interpretable metric here: it is the number a targeting
decision would actually be made on.

**The Response model and the uplift models are not the same kind of model,
and are not compared as if they were.** The Response model estimates
P(conversion | X) -- pure outcome prediction, with no reference to
treatment at all. The uplift models (T-Learner, X-Learner, Causal Forest)
estimate tau(X) = E[Y(1)-Y(0) | X] -- the causal treatment effect itself.
Below, the Response model gets its own table scored on the metrics that
actually match its objective (ROC-AUC, PR-AUC); Qini/AUUC are reported for
it only as an optional, explicitly-labeled reference point, never ranked
alongside the causal estimators in the same table.

In [ ]:
if RUN_STAGE in ("report", "all"):
    reference_label = next(iter(available))
    reference_predictions = load_parquet(available[reference_label] / "predictions.parquet")
    test_ref = reference_predictions[reference_predictions["partition"] == "test"].sort_values("row_id")
    T_test_ref, Y_test_ref = test_ref["treatment"].to_numpy(), test_ref["outcome"].to_numpy()
    rng = np.random.default_rng(SEED)
    random_metrics = evaluate_ranking(rng.uniform(size=len(T_test_ref)), T_test_ref, Y_test_ref)

    OBJECTIVES = {
        "Response LightGBM": "Outcome prediction: P(Y | X)",
        "T-Learner": "CATE estimation: tau(X) = E[Y(1)-Y(0) | X]",
        "X-Learner": "CATE estimation: tau(X) = E[Y(1)-Y(0) | X]",
        "Causal Forest": "CATE estimation: tau(X) = E[Y(1)-Y(0) | X]",
    }
    rows = []
    for label, d in available.items():
        m = load_json(d / "metrics.json")
        row = {"model": label, "objective": OBJECTIVES[label],
               "auuc_above_random": m["test_auuc_above_random"],
               "qini_above_random": m["test_qini_above_random"],
               "auuc_area": m["test_auuc_area"], "qini_area": m["test_qini_area"]}
        row.update({f"uplift@{k}": v for k, v in m["test_uplift_at_k"].items()})
        rows.append(row)
    rows.append({
        "model": "Random (reference)", "objective": "Random ranking (theoretical floor)",
        "auuc_above_random": random_metrics.auuc_above_random,
        "qini_above_random": random_metrics.qini_above_random, "auuc_area": random_metrics.auuc_area,
        "qini_area": random_metrics.qini_area,
        **{f"uplift@{k}": v for k, v in random_metrics.uplift_at_k.items()},
    })
    # Kept as an internal, all-models frame for Section 7's "did any causal
    # estimator beat the non-causal ranking" question -- never displayed as
    # a single table (see the two tables below).
    comparison = pd.DataFrame(rows).set_index("model").sort_values("qini_above_random", ascending=False)
    save_csv(comparison.reset_index(), stage_dir("report") / "model_comparison.csv")

    if "Response LightGBM" in available:
        baseline_metrics_row = load_json(available["Response LightGBM"] / "metrics.json")
        baseline_table = pd.DataFrame([{
            "Model": "Response LightGBM", "Objective": "Outcome prediction",
            "ROC-AUC (validation)": round(baseline_metrics_row["val_roc_auc"], 5),
            "PR-AUC / Average Precision (validation)": round(baseline_metrics_row["val_average_precision"], 5),
        }]).set_index("Model")
        print("Baseline model")
        display(baseline_table)
        print(
            "Reference only -- ranking users by this model's predicted conversion probability gives "
            f"qini_above_random={comparison.loc['Response LightGBM', 'qini_above_random']:.5f} and "
            f"auuc_above_random={comparison.loc['Response LightGBM', 'auuc_above_random']:.5f} on the test "
            "set. Response model Qini/AUUC does NOT represent an estimated treatment effect -- it measures "
            "the outcome-ranking baseline obtained by ranking users according to predicted conversion "
            "probability, ignoring treatment entirely. See Section 7 for what it means if this exceeds a "
            "causal estimator's score.\n"
        )

    print("Uplift model comparison")
    uplift_labels = {"T-Learner", "X-Learner", "Causal Forest", "Random (reference)"}
    uplift_table = comparison.loc[comparison.index.isin(uplift_labels),
                                    ["objective", "qini_above_random", "auuc_above_random", "uplift@10pct"]]
    uplift_table = uplift_table.rename(columns={
        "objective": "Objective", "qini_above_random": "Qini (above random)",
        "auuc_above_random": "AUUC (above random)", "uplift@10pct": "Uplift@10%",
    })
    display(uplift_table.round(5))

### Qini and uplift curves

*What does the shape mean?* A curve that rises steadily above the random
diagonal and only flattens near full coverage says the model's ranking
carries real signal deep into the population. A curve that rises briefly
then flattens early says the model has isolated a *small* responsive
subgroup but has little to say about the rest. A curve that tracks the
random line says the ranking carries no usable incremental signal at all.

In [ ]:
if RUN_STAGE in ("report", "all"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for label, d in available.items():
        qc, uc = load_csv_artifact(d / "qini_curve.csv"), load_csv_artifact(d / "uplift_curve.csv")
        ax1.plot(qc["coverage"], qc["qini_gain"], label=label)
        ax2.plot(uc["coverage"], uc["uplift_gain"], label=label)
    ax1.plot(random_metrics.qini_curve["coverage"], random_metrics.qini_curve["qini_gain"],
              "--", color="#888888", label="Random (reference)")
    ax2.plot(random_metrics.uplift_curve["coverage"], random_metrics.uplift_curve["uplift_gain"],
              "--", color="#888888", label="Random (reference)")
    ax1.set_xlabel("Coverage"); ax1.set_ylabel("Qini gain"); ax1.set_title("Qini curves (test)"); ax1.legend()
    ax2.set_xlabel("Coverage"); ax2.set_ylabel("Uplift gain"); ax2.set_title("Uplift curves / AUUC (test)"); ax2.legend()
    fig.tight_layout()
    fig.savefig(figures_dir / "qini_uplift_curves.png", dpi=120)
    plt.show()

In [ ]:
if RUN_STAGE in ("report", "all"):
    fig, ax = plt.subplots(figsize=(9, 4))
    uplift_table[["Qini (above random)", "AUUC (above random)"]].plot.barh(
        ax=ax, title="Test-set ranking performance above the random reference (uplift/CATE models only)"
    )
    fig.tight_layout()
    fig.savefig(figures_dir / "comparison_barh.png", dpi=120)
    plt.show()

### Treatment effect distribution

A model whose predicted effect is nearly constant across users is not
finding heterogeneity in the treatment effect, whatever its Qini score says
-- it has effectively collapsed to predicting the population ATE for
everyone.

In [ ]:
if RUN_STAGE in ("report", "all"):
    causal_models = [m for m in ("T-Learner", "X-Learner", "Causal Forest") if m in available]
    if causal_models:
        cate = pd.DataFrame({
            label: load_parquet(available[label] / "predictions.parquet")
                .pipe(lambda f: f[f["partition"] == "test"].sort_values("row_id"))["score"].to_numpy()
            for label in causal_models
        })
        display(cate.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T.round(6))

        fig, axes = plt.subplots(1, len(causal_models), figsize=(5 * len(causal_models), 3.5), squeeze=False)
        for ax, label in zip(axes[0], causal_models):
            ax.hist(cate[label], bins=60, color="#2b6cb0")
            ax.axvline(0.0, color="k", linestyle="--", linewidth=1)
            ax.set_title(f"{label}\npredicted CATE")
        fig.tight_layout()
        fig.savefig(figures_dir / "cate_distribution.png", dpi=120)
        plt.show()
    else:
        print("No causal model (T-Learner / X-Learner / Causal Forest) artifacts available yet.")

### Uplift decile analysis

If a ranking is real, observed uplift should decline from decile 1 (highest
predicted) down to decile 10 (lowest) -- a monotonic-looking decline is
supporting evidence beyond the single Qini number; a flat or non-monotonic
pattern says the ranking is not reliably ordering users by real incremental
effect, even if the aggregate Qini looks positive.

In [ ]:
if RUN_STAGE in ("report", "all") and causal_models:
    deciles = pd.DataFrame({
        label: load_csv_artifact(available[label] / "decile_table.csv").set_index("decile")["observed_uplift"]
        for label in causal_models
    })
    display(deciles.round(5))

    ax = deciles.plot(marker="o", figsize=(9, 4),
                       title="Observed test uplift by predicted-CATE decile (1 = highest predicted)")
    ax.axhline(0.0, color="k", linestyle="--", linewidth=1)
    ax.set_xlabel("Decile of predicted CATE"); ax.set_ylabel("Observed uplift")
    ax.figure.savefig(figures_dir / "uplift_deciles.png", dpi=120)
    plt.show()

    if len(causal_models) > 1:
        print("Spearman rank correlation between causal models' predicted CATE:")
        print(cate.corr(method="spearman").round(3))
        print("Interpretation: where the causal estimators agree on who is high/low uplift, that signal "
              "is more plausibly real than where only one model sees it -- agreement is not proof, but "
              "disagreement is a caution flag against over-trusting any single model's ranking.")

### Response model diagnostics (ROC / PR / calibration)

Reproduced here from the baseline stage's saved artifacts so the report
stage stays self-contained after a kernel restart.

In [ ]:
if RUN_STAGE in ("report", "all"):
    if "Response LightGBM" in available:
        baseline_dir = available["Response LightGBM"]
        roc = load_csv_artifact(baseline_dir / "roc_curve.csv")
        pr = load_csv_artifact(baseline_dir / "pr_curve.csv")
        cal = load_csv_artifact(baseline_dir / "calibration_curve.csv")
        baseline_metrics = load_json(baseline_dir / "metrics.json")

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].plot(roc["fpr"], roc["tpr"], color="#2b6cb0")
        axes[0].plot([0, 1], [0, 1], "--", color="#888888")
        axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
        axes[0].set_title(f"ROC (AUC={baseline_metrics['val_roc_auc']:.4f})")
        axes[1].plot(pr["recall"], pr["precision"], color="#2b6cb0")
        axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
        axes[1].set_title(f"PR (AP={baseline_metrics['val_average_precision']:.4f})")
        axes[2].plot(cal["mean_predicted"], cal["fraction_positive"], "o-", color="#2b6cb0")
        axes[2].plot([0, 1], [0, 1], "--", color="#888888")
        axes[2].set_xlabel("Mean predicted probability"); axes[2].set_ylabel("Observed conversion rate")
        axes[2].set_title("Calibration (validation)")
        fig.suptitle("Response model diagnostics -- validation partition")
        fig.tight_layout()
        fig.savefig(figures_dir / "baseline_diagnostics.png", dpi=120)
        plt.show()
    else:
        print("Response model artifacts not available.")

## Section 7 -- Model Interpretation

This section is the point of the whole comparison -- not just naming a
winner, but explaining, in light of everything above, *why* the results came
out the way they did. The interpretation below is generated from this run's
actual numbers, not written in advance -- the branches exist because either
outcome (a causal model wins, or the plain response model wins) is a real,
informative possibility on a dataset with a rare outcome and a genuinely
uncertain treatment effect.

In [ ]:
if RUN_STAGE in ("report", "all"):
    causal_models_all = [m for m in ("T-Learner", "X-Learner", "Causal Forest") if m in available]
    ranked = comparison.drop(index="Random (reference)", errors="ignore")
    best = ranked.index[0]
    best_qini = ranked.loc[best, "qini_above_random"]
    random_qini = random_metrics.qini_above_random
    beat_random = [m for m in ranked.index if ranked.loc[m, "qini_above_random"] > random_qini]

    print("BEST MODEL:", best)
    print(f"  Qini above random : {best_qini:.5f}")
    print(f"  AUUC above random : {ranked.loc[best, 'auuc_above_random']:.5f}")
    print(f"  uplift@10pct      : {ranked.loc[best, 'uplift@10pct']}")
    print()
    print("Models beating the random reference:", beat_random or "NONE")
    print()

    if best_qini <= random_qini:
        print(
            "No model beat the random-ranking reference on this run. Given the outcome rarity shown in "
            f"Section 3 ({dataset_summary['conversion_rate']:.3%} overall conversion), this is a real "
            "possible outcome, not necessarily a bug: with this few positive labels per arm, even a "
            "correctly-implemented estimator can fail to find a statistically detectable ranking signal. "
            "Treat every claim below as provisional on more data or a stronger treatment effect, not as "
            "evidence the methodology is wrong."
        )
    elif best == "Response LightGBM":
        print(
            "The non-causal Response model ranked best on this run. Read this carefully: it means "
            "P(conversion | X) alone was more predictive of the *ranking* Qini rewards than any explicit "
            "treatment-effect estimate was. This can genuinely happen when the treatment effect is fairly "
            "homogeneous across users (so there is little true tau(X) heterogeneity for a causal model to "
            "find) and/or when conversion propensity is correlated enough with treatment response that a "
            "response-based ranking incidentally approximates a causal one. It does NOT mean predictive "
            "performance equals causal effect estimation in general -- it means that, on this specific "
            "dataset and split, the gap between the two questions happened to be small. The response "
            "model still cannot distinguish User A (persuadable) from User B (would convert anyway) from "
            "Section 0 -- it simply got lucky that, in aggregate, ranking by conversion propensity moved "
            "persuadable users upward too."
        )
    else:
        print(
            f"A causal estimator ({best}) ranked best, ahead of the non-causal Response model "
            f"({ranked.loc['Response LightGBM', 'qini_above_random'] if 'Response LightGBM' in ranked.index else float('nan'):.5f} "
            "Qini above random). This is the result uplift modeling is meant to produce: a ranking that "
            "uses the treatment/control contrast, not conversion propensity alone, to separate "
            "persuadable users from sure-things and lost causes."
        )

    underperformed = [m for m in causal_models_all if m != best and ranked.loc[m, "qini_above_random"] < best_qini]
    if underperformed:
        print()
        print(f"Why {', '.join(underperformed)} may have under-performed {best}:")
        print(
            "  - Rare conversion (Section 3) means every model is estimating a small difference between "
            "two already-small rates -- the effective sample size for the causal signal is much smaller "
            "than the raw row count suggests.\n"
            "  - The true treatment effect may simply be weak or fairly homogeneous across X here, leaving "
            "little heterogeneity for any estimator to recover -- a hard problem is a property of the "
            "data, not evidence a specific implementation is broken.\n"
            "  - More flexible estimators (Causal Forest in particular) trade this away for higher "
            "variance: with fewer effective positive outcomes to split on, a high-capacity model can rank "
            "*less* reliably than a more constrained one, not more.\n"
            "  - Causal Forest specifically also works from a coarser categorical representation (K=8 "
            "top-K encoding vs. the other models' full-cardinality LightGBM categorical splits) -- a "
            "resource-driven tradeoff (Section 4.4), not a performance-tuned one."
        )

### Causal Forest specifically

In [ ]:
if RUN_STAGE in ("report", "all"):
    if "Causal Forest" in available:
        cf_metrics = load_json(available["Causal Forest"] / "metrics.json")
        cf_deciles = load_csv_artifact(available["Causal Forest"] / "decile_table.csv").sort_values("decile")
        cf_resource = load_json(stage_dir("causal_forest") / "resource_evidence.json")
        cf_cate = cate["Causal Forest"]
        top_decile, bottom_decile = cf_deciles.iloc[0], cf_deciles.iloc[-1]
        monotonic = (cf_deciles["observed_uplift"].diff().dropna() <= 1e-9).all()

        print(f"Test qini_above_random  : {cf_metrics['test_qini_above_random']:.5f}")
        print(f"Mean / median predicted CATE : {cf_cate.mean():.6f} / {cf_cate.median():.6f}")
        print(f"Share of rows with predicted CATE > 0 : {(cf_cate > 0).mean():.3%}")
        print(f"Top decile (highest predicted) observed uplift    : {top_decile['observed_uplift']:.5f}  (n={int(top_decile['n'])})")
        print(f"Bottom decile (lowest predicted) observed uplift  : {bottom_decile['observed_uplift']:.5f}  (n={int(bottom_decile['n'])})")
        print(f"Categorical representation: K={cf_resource['categorical_top_k']} -> "
              f"{cf_resource['encoded_feature_count']} encoded columns, max_depth={cf_resource['max_depth']} -- "
              "coarser than the other estimators' full-cardinality LightGBM categorical splits and a capped "
              "tree depth, both resource-driven choices (see README's Methodology notes).")
        print(f"Deciles monotonically declining from 1 to 10: {monotonic} -- "
              f"{'supporting' if monotonic else 'not clearly supporting'} the ranking as more than noise.")
    else:
        print(
            "Not available -- Causal Forest resource gate/model execution is pending. "
            "Set RUN_CAUSAL_FOREST = True, RUN_STAGE in {'causal_forest', 'all'}, and re-run."
        )

## Section 8 -- Limitations

- **Anonymized features prevent semantic interpretation.** `f0`-`f11` have no
  documented business meaning; every statement in Sections 1-2 is about
  statistical behavior only, never "what a feature represents."
- **Rare outcome, weak/uncertain uplift signal.** Conversion is rare
  (Section 3); the true treatment effect heterogeneity this dataset contains
  is unknown, and every model here is estimating a difference of two small
  rates from a finite sample -- see Section 7 for what this implies about
  interpreting any single model's ranking.
- **Offline evaluation only.** Qini/AUUC are computed against the historical
  A/B test's logged outcomes -- there is no online experiment confirming that
  acting on these rankings would reproduce the measured incremental effect.
- **Point estimates only; uncertainty in CATE estimation is not quantified.**
  No confidence intervals on the metric differences here; a paired,
  arm-stratified bootstrap over the fixed test predictions would be the
  natural next addition.
- **Computational constraints shaped part of the design, not just the
  science.** The Causal Forest's `K=8` categorical cap and `max_depth=20`
  safety cap are both memory/runtime tradeoffs (Section 4.4), not
  performance-tuned choices, and give it a coarser representation and a
  bounded tree size relative to the other estimators.
- **Not a validated causal mechanism.** Predicted CATE is not a true
  individual treatment effect -- both potential outcomes are never observed
  for any one row, which is also why no PEHE against ground truth is
  reported.
- **Specific to this dataset and these implementations.** One dataset, one
  implementation of each method, one hyperparameter setting -- a valid
  conclusion has the form "estimator A ranked incremental conversion better
  than estimator B *here*," not a universal claim about either method.

## Section 9 -- Conclusion

**What was learned:** Section 7's printed output states, from this run's own
numbers, which estimator ranked test-set incremental conversion best under
this protocol, and whether it beat the random-targeting floor at all.

**What the result means:** if a causal estimator won, it means using the
treatment/control contrast produced a better targeting ranking than
conversion propensity alone -- the central premise of uplift modeling held
up empirically here. If the Response model won instead, it means that, on
this dataset and split, the gap between "who converts" and "who converts
*because of* treatment" was smaller than hoped, which is itself a legitimate
and reportable finding, not a failed experiment.

**What cannot be concluded:** that the winning estimator is universally
best (one dataset, one implementation of each, one hyperparameter setting);
that a meta-learner family is intrinsically superior to Causal Forest or vice
versa; that predicted CATE is a true individual treatment effect; or that
acting on these rankings in production would reproduce the measured effect
without an online experiment. See Section 8 for the full list.

**Next steps:** a paired bootstrap confidence interval on the Qini gap
between the top two models; an online holdout experiment on the model's
actual targeting decisions; and, if resources allow, a measured
memory/runtime benchmark at `K=16`/`K=32` for the Causal Forest encoding.

In [ ]:
if RUN_STAGE in ("report", "all"):
    print("=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    print(f"Best performing model : {best}")
    print(f"  Qini above random   : {ranked.loc[best, 'qini_above_random']:.5f}")
    print(f"  AUUC above random   : {ranked.loc[best, 'auuc_above_random']:.5f}")
    print(f"  uplift@10pct        : {ranked.loc[best, 'uplift@10pct']}")
    print(f"Models beating random  : {beat_random or 'NONE'}")
    print(f"Models evaluated       : {list(available)}")
    missing = [m for m in MODEL_STAGE_DIRS if m not in available]
    print(f"Models not yet run     : {missing or 'none'}")
    print(f"Report artifacts       : {stage_dir('report')}")

## Appendix -- Full configuration

Every value below is consumed by `src/` or this notebook and is kept in sync
with `configs/config.yaml` by `tests/test_config.py`; it is reproduced here
in full for a technical reader who wants every parameter, rather than in the
main narrative above.

In [ ]:
print("seed:", CONFIG["seed"])
for section in ("split", "lightgbm", "xlearner", "causal_forest", "evaluation"):
    print(f"\n{section}:")
    for key, value in CONFIG[section].items():
        print(f"  {key}: {value}")
print("\nRun environment (this session):")
for key, value in ENV_INFO.items():
    print(f"  {key}: {value}")